# YOLO26s — Continue Existing 35-Epoch VisDrone Run to 45 Epochs
## Google Colab L4 — Final continuation notebook

This notebook was built specifically for the already-completed YOLO26s run found in the supplied notebook.

### Audited source run
- Model: **YOLO26s**
- Old experiment ID: `main_v1.12`
- Old run: `main_1280`
- Old training budget: **35 epochs**
- Input: **1280 × 1280**
- Batch: **8**
- Optimizer: **AdamW**
- Seed: **42**
- Save period: **1 epoch**
- Old persistent run:
  `/content/drive/MyDrive/aerial_person_final_product/runs/aerial_yolo26s/main_v1.12/main_1280`
- The supplied notebook recorded **35/35 history rows** and a best validation `mAP50-95 ≈ 0.34058`.

### Important scientific note
The old YOLO26s run was **not trained on only the 6471 official VisDrone train images**.  
It used the project's ATPC recipe:

- 6471 official VisDrone training images
- + 2800 deterministic tiny-context training tiles
- = **9271 training images**
- Validation remained the official **548-image VisDrone validation split**

Therefore this 35+10 continuation is internally consistent with the old YOLO26s experiment, but it is **not a strict same-training-set baseline** against the clean RT-DETR and BPD runs if those runs use only the 6471 official train images.

### What this notebook does
1. Audits the old 35-epoch Drive artifacts.
2. Rebuilds the same person-only VisDrone + ATPC training recipe locally.
3. Prefers the latest resumable epoch checkpoint from the old run.
4. If a resumable optimizer checkpoint exists, it migrates the run to a **new Drive path** and continues epochs **36–45** with optimizer state.
5. If the completed old run has only stripped checkpoints, it safely fine-tunes the old `last.pt` for 10 additional finishing epochs.
6. Preserves and exports the **complete 45-epoch history**.
7. Prints a clean report after each new epoch.
8. Saves final validation, COCO AP50/AP75/AP-S/AP-M/AP-L/AR100, plots, hashes, and final checkpoints.

The private independent Final Test is not used here.

## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("Google Drive mounted.")

MessageError: Error: credential propagation was unsuccessful

## 2. Reproducible environment

The old run used `ultralytics==8.4.114`.  
This notebook pins the same Ultralytics version and deliberately does not upgrade PyTorch, Torchvision, NumPy, or SciPy.

In [ ]:
import subprocess
import sys

import numpy as np
import scipy
import torch
import torchvision

print("NumPy      :", np.__version__)
print("SciPy      :", scipy.__version__)
print("PyTorch    :", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA       :", torch.version.cuda)

packages = [
    "ultralytics==8.4.114",
    "pycocotools>=2.0.8",
    "PyYAML>=6.0",
    "tqdm>=4.66",
    "pillow>=10.0",
    "pandas>=2.0",
    "matplotlib>=3.7",
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        *packages,
    ]
)

import ultralytics

print("\nUltralytics:", ultralytics.__version__)
print("Environment setup: PASSED")

## 3. Imports and fixed paths

In [ ]:
from __future__ import annotations

import csv
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import zipfile

from collections import deque
from pathlib import Path
from typing import Any, Sequence

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.data.utils import img2label_paths
from ultralytics.utils import ASSETS_URL

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


# ============================================================
# OLD 35-EPOCH RUN — READ ONLY
# ============================================================

OLD_EXPERIMENT_ROOT = Path(
    "/content/drive/MyDrive/"
    "aerial_person_final_product/"
    "runs/aerial_yolo26s/main_v1.12"
)

OLD_RUN_DIR = OLD_EXPERIMENT_ROOT / "main_1280"
OLD_WEIGHTS_DIR = OLD_RUN_DIR / "weights"
OLD_RESULTS_CSV = OLD_RUN_DIR / "results.csv"
OLD_ARGS_YAML = OLD_RUN_DIR / "args.yaml"
OLD_LAST = OLD_WEIGHTS_DIR / "last.pt"
OLD_BEST = OLD_WEIGHTS_DIR / "best.pt"


# ============================================================
# NEW PERSISTENT OUTPUT — DIFFERENT FROM THE OLD RUN
# ============================================================

NEW_ROOT = Path(
    "/content/drive/MyDrive/"
    "aerial_human_detection/step3/"
    "yolo26s_visdrone_person_45e_1280_from_35e"
)

NEW_RUN_DIR = NEW_ROOT / "combined_run_45e"
NEW_WEIGHTS_DIR = NEW_RUN_DIR / "weights"

SOURCE_SNAPSHOT_DIR = NEW_ROOT / "source_35e_snapshot"
HISTORY_DIR = NEW_ROOT / "history"
REPORTS_DIR = NEW_ROOT / "reports"
METRICS_DIR = NEW_ROOT / "metrics"
FINAL_WEIGHTS_DIR = NEW_ROOT / "final_weights"

for directory in [
    NEW_ROOT,
    NEW_RUN_DIR,
    NEW_WEIGHTS_DIR,
    SOURCE_SNAPSHOT_DIR,
    HISTORY_DIR,
    REPORTS_DIR,
    METRICS_DIR,
    FINAL_WEIGHTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOCAL DATASET WORKSPACE
# Keep the historical local dataset path for compatibility.
# ============================================================

LOCAL_PROJECT_ROOT = Path("/content/aerial_person_final_product")
DATASET_ROOT = (
    LOCAL_PROJECT_ROOT
    / "datasets"
    / "visdrone_person_context_tiles"
)

RAW_ROOT = Path("/content/aerial_yolo26s_45e/raw")


# ============================================================
# FIXED CONTINUATION CONTROLS
# ============================================================

OLD_EPOCHS = 35
ADDITIONAL_EPOCHS = 10
TOTAL_EPOCHS = 45

IMAGE_SIZE = 1280
BATCH_SIZE = 8
WORKERS = 4
SEED = 42
SAVE_PERIOD = 1
USE_AMP = True
REQUIRE_L4 = True

ATPC_CONTEXT_TILES = 2800
TINY_SIDE_THRESHOLD_PX = 24.0
CONTEXT_TILE_MIN_SIDE = 320
CONTEXT_TILE_MAX_SIDE = 704

print("Old run :", OLD_RUN_DIR)
print("New root:", NEW_ROOT)
print("Target  :", f"{TOTAL_EPOCHS} total epochs")
print("Final Test used: False")

## 4. Verify NVIDIA L4

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select an NVIDIA GPU runtime in Colab."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print("GPU :", gpu_name)
print(f"VRAM: {vram_gb:.2f} GB")
subprocess.run(["nvidia-smi"], check=False)

if REQUIRE_L4 and "L4" not in gpu_name.upper():
    raise RuntimeError(
        f"This controlled continuation expects NVIDIA L4, but found: {gpu_name}"
    )

print("GPU verification: PASSED")

## 5. Audit the previous 35-epoch Drive run

Training will not proceed unless the old `results.csv`, `args.yaml`, `best.pt`, and `last.pt` are found and the old history contains exactly 35 rows.

In [ ]:
def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        while True:
            chunk = stream.read(block_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


for required in [
    OLD_RESULTS_CSV,
    OLD_ARGS_YAML,
    OLD_LAST,
    OLD_BEST,
]:
    if not required.exists():
        raise FileNotFoundError(
            f"Required old-run artifact was not found:\n{required}"
        )


old_history = pd.read_csv(OLD_RESULTS_CSV)
old_history.columns = [str(c).strip() for c in old_history.columns]

if len(old_history) != OLD_EPOCHS:
    raise RuntimeError(
        f"The source history has {len(old_history)} rows; "
        f"exactly {OLD_EPOCHS} are required."
    )


old_args = yaml.safe_load(
    OLD_ARGS_YAML.read_text(encoding="utf-8")
)

checks = {
    "epochs": int(old_args.get("epochs", -1)),
    "imgsz": int(old_args.get("imgsz", -1)),
    "batch": int(old_args.get("batch", -1)),
    "seed": int(old_args.get("seed", -1)),
    "save_period": int(old_args.get("save_period", -1)),
    "optimizer": str(old_args.get("optimizer")),
}

if checks["epochs"] != 35:
    raise RuntimeError(f"Old run epochs mismatch: {checks['epochs']}")

if checks["imgsz"] != IMAGE_SIZE:
    raise RuntimeError(f"Old run image size mismatch: {checks['imgsz']}")

if checks["batch"] != BATCH_SIZE:
    raise RuntimeError(f"Old run batch mismatch: {checks['batch']}")

if checks["seed"] != SEED:
    raise RuntimeError(f"Old run seed mismatch: {checks['seed']}")

if checks["save_period"] != 1:
    raise RuntimeError(
        f"Old run save_period was {checks['save_period']}; expected 1."
    )


map_col = next(
    (
        c
        for c in old_history.columns
        if "map50-95" in c.lower()
    ),
    None,
)

if map_col is None:
    raise RuntimeError(
        "Could not find mAP50-95 in the old results.csv."
    )

old_best_map = float(
    pd.to_numeric(
        old_history[map_col],
        errors="coerce",
    ).max()
)

print("=" * 88)
print("OLD YOLO26s RUN AUDIT: PASSED")
print("=" * 88)
print("History rows      :", len(old_history))
print("Image size        :", checks["imgsz"])
print("Batch             :", checks["batch"])
print("Optimizer         :", checks["optimizer"])
print("Seed              :", checks["seed"])
print("Save period       :", checks["save_period"])
print("Best old mAP50-95 :", f"{old_best_map:.6f}")
print("Old best SHA256   :", sha256_file(OLD_BEST))
print("Old last SHA256   :", sha256_file(OLD_LAST))
print("=" * 88)


# Persist immutable copies of the original 35-epoch history and metadata.
old_history.to_csv(
    HISTORY_DIR / "epochs_001_035_original.csv",
    index=False,
)

shutil.copy2(
    OLD_ARGS_YAML,
    SOURCE_SNAPSHOT_DIR / "args_35e_original.yaml",
)

shutil.copy2(
    OLD_RESULTS_CSV,
    SOURCE_SNAPSHOT_DIR / "results_35e_original.csv",
)

# Keep source best/last snapshots under unique names.
shutil.copy2(
    OLD_BEST,
    SOURCE_SNAPSHOT_DIR / "best_35e_original.pt",
)

shutil.copy2(
    OLD_LAST,
    SOURCE_SNAPSHOT_DIR / "last_35e_original.pt",
)

source_audit = {
    "old_run_dir": str(OLD_RUN_DIR),
    "history_rows": len(old_history),
    "old_best_map50_95": old_best_map,
    "old_best_sha256": sha256_file(OLD_BEST),
    "old_last_sha256": sha256_file(OLD_LAST),
    "old_args": checks,
    "target_total_epochs": TOTAL_EPOCHS,
    "new_root": str(NEW_ROOT),
}

(
    SOURCE_SNAPSHOT_DIR
    / "source_35e_audit.json"
).write_text(
    json.dumps(
        source_audit,
        indent=2,
    ),
    encoding="utf-8",
)

## 6. Find the best resumable source checkpoint

Because the old run completed all 35 epochs, Ultralytics may have stripped optimizer state from `last.pt` during finalization.

Since the old run used `save_period=1`, this notebook first searches the old `epoch*.pt` files for the latest checkpoint that still contains optimizer state.

Priority:
1. Latest resumable `epoch*.pt`
2. `last.pt` if it is resumable
3. Controlled 10-epoch finishing fine-tune from the old `last.pt`

In [ ]:
def checkpoint_state_summary(path: Path) -> dict[str, Any]:
    state = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    summary = {
        "path": str(path),
        "epoch": int(state.get("epoch", -1)),
        "has_optimizer": state.get("optimizer") is not None,
        "has_train_args": isinstance(
            state.get("train_args"),
            dict,
        ),
    }

    del state
    gc.collect()

    return summary


def epoch_checkpoint_number(path: Path) -> int:
    match = re.search(
        r"epoch(\d+)",
        path.stem,
        flags=re.IGNORECASE,
    )

    return (
        int(match.group(1))
        if match
        else -1
    )


epoch_candidates = sorted(
    OLD_WEIGHTS_DIR.glob("epoch*.pt"),
    key=epoch_checkpoint_number,
    reverse=True,
)

candidate_paths = list(epoch_candidates[:5])

if OLD_LAST not in candidate_paths:
    candidate_paths.append(OLD_LAST)


candidate_summaries = []

for candidate in candidate_paths:
    try:
        summary = checkpoint_state_summary(candidate)
        candidate_summaries.append(summary)

        print(
            Path(summary["path"]).name,
            "| epoch=",
            summary["epoch"],
            "| optimizer=",
            summary["has_optimizer"],
        )

    except Exception as exc:
        print(
            "Could not inspect",
            candidate,
            ":",
            exc,
        )


RESUMABLE_SOURCE = None

for summary in candidate_summaries:
    if (
        summary["epoch"] >= OLD_EPOCHS - 1
        and summary["has_optimizer"]
    ):
        RESUMABLE_SOURCE = Path(summary["path"])
        break


if RESUMABLE_SOURCE is not None:
    CONTINUATION_MODE = "exact_optimizer_resume"
    SOURCE_CHECKPOINT = RESUMABLE_SOURCE
else:
    CONTINUATION_MODE = "controlled_10e_finetune_fallback"
    SOURCE_CHECKPOINT = OLD_LAST


print("\n" + "=" * 88)
print("CONTINUATION MODE")
print("=" * 88)
print("Mode             :", CONTINUATION_MODE)
print("Source checkpoint:", SOURCE_CHECKPOINT)
print("=" * 88)


(
    SOURCE_SNAPSHOT_DIR
    / "checkpoint_selection.json"
).write_text(
    json.dumps(
        {
            "mode": CONTINUATION_MODE,
            "source_checkpoint": str(SOURCE_CHECKPOINT),
            "candidates": candidate_summaries,
        },
        indent=2,
    ),
    encoding="utf-8",
)

## 7. Robustly download VisDrone train/val

The previous YOLO26s run used the official VisDrone data.  
This downloader is retryable and does not depend on a one-shot Ultralytics curl download.

In [ ]:
RAW_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_URL = (
    f"{ASSETS_URL}/VisDrone2019-DET-train.zip"
)

VAL_URL = (
    f"{ASSETS_URL}/VisDrone2019-DET-val.zip"
)


def count_images(directory: Path) -> int:
    extensions = {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".tif",
        ".tiff",
        ".webp",
    }

    if not directory.exists():
        return 0

    return sum(
        1
        for p in directory.rglob("*")
        if p.is_file()
        and p.suffix.lower() in extensions
    )


def locate_native_split(
    root: Path,
    expected_count: int,
) -> tuple[Path, Path] | None:
    candidates = []

    for images_dir in root.rglob("images"):
        ann_dir = images_dir.parent / "annotations"

        if not ann_dir.is_dir():
            continue

        count = count_images(images_dir)

        if count:
            candidates.append(
                (
                    abs(count - expected_count),
                    -count,
                    images_dir,
                    ann_dir,
                )
            )

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: (
            x[0],
            x[1],
        )
    )

    _, _, images_dir, ann_dir = candidates[0]

    if count_images(images_dir) != expected_count:
        return None

    return (
        images_dir.resolve(),
        ann_dir.resolve(),
    )


def robust_download_and_extract(
    url: str,
    zip_name: str,
    expected_count: int,
):
    existing = locate_native_split(
        RAW_ROOT,
        expected_count,
    )

    if existing is not None:
        print(
            f"{zip_name}: already extracted and verified "
            f"({expected_count} images)."
        )
        return existing

    zip_path = RAW_ROOT / zip_name
    part_path = RAW_ROOT / f"{zip_name}.part"

    for attempt in range(1, 9):
        print(
            f"\nDownload attempt {attempt}/8: {zip_name}"
        )

        command = [
            "wget",
            "--continue",
            "--tries=4",
            "--timeout=60",
            "--read-timeout=60",
            "--progress=bar:force:noscroll",
            "-O",
            str(part_path),
            url,
        ]

        result = subprocess.run(
            command,
        )

        if result.returncode != 0:
            print(
                "wget failed with code",
                result.returncode,
            )
            time.sleep(
                min(
                    30,
                    attempt * 4,
                )
            )
            continue

        try:
            with zipfile.ZipFile(part_path, "r") as archive:
                bad = archive.testzip()

                if bad is not None:
                    raise RuntimeError(
                        f"Corrupt member: {bad}"
                    )

                archive.extractall(
                    RAW_ROOT
                )

            part_path.replace(
                zip_path
            )

        except Exception as exc:
            print(
                "ZIP verification/extraction failed:",
                exc,
            )
            time.sleep(
                min(
                    30,
                    attempt * 4,
                )
            )
            continue

        located = locate_native_split(
            RAW_ROOT,
            expected_count,
        )

        if located is not None:
            print(
                f"{zip_name}: extraction verified "
                f"({expected_count} images)."
            )
            return located

        print(
            "Extraction completed but expected split "
            "could not be verified."
        )

    raise RuntimeError(
        f"Could not download/extract {zip_name} after 8 attempts."
    )


TRAIN_NATIVE_IMAGES, TRAIN_NATIVE_ANN = (
    robust_download_and_extract(
        TRAIN_URL,
        "VisDrone2019-DET-train.zip",
        6471,
    )
)

VAL_NATIVE_IMAGES, VAL_NATIVE_ANN = (
    robust_download_and_extract(
        VAL_URL,
        "VisDrone2019-DET-val.zip",
        548,
    )
)

print("\nTrain images:", TRAIN_NATIVE_IMAGES)
print("Train ann   :", TRAIN_NATIVE_ANN)
print("Val images  :", VAL_NATIVE_IMAGES)
print("Val ann     :", VAL_NATIVE_ANN)

## 8. Rebuild the old person-only + ATPC recipe

This reproduces the old YOLO26s training recipe:
- native VisDrone `pedestrian` + `people` → `person`
- official 6471 train images
- 2800 deterministic ATPC tiny-context tiles
- official 548 validation images
- no ATPC augmentation in validation

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


def parse_visdrone_line(
    line: str,
):
    parts = [
        x.strip()
        for x in line.strip().split(",")
    ]

    if len(parts) < 8:
        return None

    try:
        x, y, w, h = map(
            float,
            parts[:4],
        )

        score = int(
            float(
                parts[4]
            )
        )

        category = int(
            float(
                parts[5]
            )
        )

        truncation = int(
            float(
                parts[6]
            )
        )

        occlusion = int(
            float(
                parts[7]
            )
        )

    except Exception:
        return None

    return (
        x,
        y,
        w,
        h,
        score,
        category,
        truncation,
        occlusion,
    )


def native_person_boxes(
    annotation_path: Path,
    image_width: int,
    image_height: int,
):
    boxes = []

    if not annotation_path.exists():
        return boxes

    for raw in annotation_path.read_text(
        encoding="utf-8-sig"
    ).splitlines():
        parsed = parse_visdrone_line(
            raw
        )

        if parsed is None:
            continue

        (
            x,
            y,
            w,
            h,
            score,
            category,
            truncation,
            occlusion,
        ) = parsed

        if score == 0:
            continue

        # Native VisDrone:
        # 1 = pedestrian
        # 2 = people
        if category not in {
            1,
            2,
        }:
            continue

        x1 = max(
            0.0,
            min(
                image_width,
                x,
            ),
        )

        y1 = max(
            0.0,
            min(
                image_height,
                y,
            ),
        )

        x2 = max(
            0.0,
            min(
                image_width,
                x + w,
            ),
        )

        y2 = max(
            0.0,
            min(
                image_height,
                y + h,
            ),
        )

        bw = x2 - x1
        bh = y2 - y1

        if (
            bw <= 0
            or bh <= 0
        ):
            continue

        cx = (
            (x1 + x2)
            / 2
            / image_width
        )

        cy = (
            (y1 + y2)
            / 2
            / image_height
        )

        nw = (
            bw
            / image_width
        )

        nh = (
            bh
            / image_height
        )

        boxes.append(
            (
                cx,
                cy,
                nw,
                nh,
            )
        )

    return boxes


def write_yolo_boxes(
    path: Path,
    boxes,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    lines = [
        "0 "
        + " ".join(
            f"{value:.8f}"
            for value in box
        )
        for box in boxes
    ]

    path.write_text(
        "\n".join(lines)
        + (
            "\n"
            if lines
            else ""
        ),
        encoding="utf-8",
    )


def stable_name(
    path: Path,
    prefix: str = "",
):
    digest = hashlib.sha1(
        str(path).encode(
            "utf-8"
        )
    ).hexdigest()[:10]

    return (
        f"{prefix}"
        f"{digest}_"
        f"{path.name}"
    )


def hardlink_or_copy(
    source: Path,
    destination: Path,
):
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if (
        destination.exists()
        or destination.is_symlink()
    ):
        destination.unlink()

    try:
        os.link(
            source,
            destination,
        )

        return "hardlink"

    except OSError:
        shutil.copy2(
            source,
            destination,
        )

        return "copy"


def normalized_to_xyxy(
    box,
    width,
    height,
):
    x, y, w, h = box

    return (
        (x - w / 2) * width,
        (y - h / 2) * height,
        (x + w / 2) * width,
        (y + h / 2) * height,
    )


def clipped_crop_box(
    center_x,
    center_y,
    side,
    width,
    height,
):
    side = int(
        min(
            side,
            width,
            height,
        )
    )

    x1 = int(
        round(
            center_x
            - side / 2
        )
    )

    y1 = int(
        round(
            center_y
            - side / 2
        )
    )

    x1 = max(
        0,
        min(
            x1,
            width - side,
        ),
    )

    y1 = max(
        0,
        min(
            y1,
            height - side,
        ),
    )

    return (
        x1,
        y1,
        x1 + side,
        y1 + side,
    )


def boxes_inside_crop(
    boxes,
    image_width,
    image_height,
    crop,
    minimum_retained_area=0.50,
):
    (
        crop_x1,
        crop_y1,
        crop_x2,
        crop_y2,
    ) = crop

    crop_width = (
        crop_x2
        - crop_x1
    )

    crop_height = (
        crop_y2
        - crop_y1
    )

    output = []

    for box in boxes:
        (
            x1,
            y1,
            x2,
            y2,
        ) = normalized_to_xyxy(
            box,
            image_width,
            image_height,
        )

        original_area = (
            max(
                0.0,
                x2 - x1,
            )
            * max(
                0.0,
                y2 - y1,
            )
        )

        if original_area <= 0:
            continue

        clipped_x1 = max(
            x1,
            crop_x1,
        )

        clipped_y1 = max(
            y1,
            crop_y1,
        )

        clipped_x2 = min(
            x2,
            crop_x2,
        )

        clipped_y2 = min(
            y2,
            crop_y2,
        )

        retained_area = (
            max(
                0.0,
                clipped_x2
                - clipped_x1,
            )
            * max(
                0.0,
                clipped_y2
                - clipped_y1,
            )
        )

        if (
            retained_area
            / original_area
            < minimum_retained_area
        ):
            continue

        clipped_x1 -= crop_x1
        clipped_y1 -= crop_y1
        clipped_x2 -= crop_x1
        clipped_y2 -= crop_y1

        center_x = (
            (
                clipped_x1
                + clipped_x2
            )
            / 2
            / crop_width
        )

        center_y = (
            (
                clipped_y1
                + clipped_y2
            )
            / 2
            / crop_height
        )

        box_width = (
            (
                clipped_x2
                - clipped_x1
            )
            / crop_width
        )

        box_height = (
            (
                clipped_y2
                - clipped_y1
            )
            / crop_height
        )

        if (
            box_width > 0
            and box_height > 0
        ):
            output.append(
                (
                    center_x,
                    center_y,
                    box_width,
                    box_height,
                )
            )

    return output


def create_context_tile(
    source_image: Path,
    source_boxes,
    selected_box_index: int,
    destination_image: Path,
    destination_label: Path,
    rng: random.Random,
):
    image = cv2.imread(
        str(
            source_image
        )
    )

    if image is None:
        return None

    image_height, image_width = (
        image.shape[:2]
    )

    selected_box = (
        source_boxes[
            selected_box_index
        ]
    )

    x, y, w, h = selected_box

    box_width_px = (
        w
        * image_width
    )

    box_height_px = (
        h
        * image_height
    )

    target_side = int(
        max(
            box_width_px,
            box_height_px,
        )
        * 22
    )

    crop_side = int(
        np.clip(
            target_side,
            CONTEXT_TILE_MIN_SIDE,
            min(
                CONTEXT_TILE_MAX_SIDE,
                image_width,
                image_height,
            ),
        )
    )

    jitter = (
        0.12
        * crop_side
    )

    center_x = (
        x
        * image_width
        + rng.uniform(
            -jitter,
            jitter,
        )
    )

    center_y = (
        y
        * image_height
        + rng.uniform(
            -jitter,
            jitter,
        )
    )

    crop = clipped_crop_box(
        center_x,
        center_y,
        crop_side,
        image_width,
        image_height,
    )

    transformed_boxes = boxes_inside_crop(
        source_boxes,
        image_width,
        image_height,
        crop,
        minimum_retained_area=0.50,
    )

    if not transformed_boxes:
        return None

    (
        crop_x1,
        crop_y1,
        crop_x2,
        crop_y2,
    ) = crop

    cropped_image = image[
        crop_y1:crop_y2,
        crop_x1:crop_x2,
    ]

    destination_image.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    ok = cv2.imwrite(
        str(
            destination_image
        ),
        cropped_image,
        [
            int(
                cv2.IMWRITE_JPEG_QUALITY
            ),
            95,
        ],
    )

    if not ok:
        return None

    write_yolo_boxes(
        destination_label,
        transformed_boxes,
    )

    return {
        "person_boxes": len(
            transformed_boxes
        ),
        "crop_side": crop_side,
    }


def build_dataset():
    if DATASET_ROOT.exists():
        shutil.rmtree(
            DATASET_ROOT
        )

    train_images_out = (
        DATASET_ROOT
        / "images"
        / "train"
    )

    train_labels_out = (
        DATASET_ROOT
        / "labels"
        / "train"
    )

    val_images_out = (
        DATASET_ROOT
        / "images"
        / "val"
    )

    val_labels_out = (
        DATASET_ROOT
        / "labels"
        / "val"
    )

    for p in [
        train_images_out,
        train_labels_out,
        val_images_out,
        val_labels_out,
    ]:
        p.mkdir(
            parents=True,
            exist_ok=True,
        )

    rows = []
    train_candidates = []

    split_specs = [
        (
            "train",
            TRAIN_NATIVE_IMAGES,
            TRAIN_NATIVE_ANN,
            train_images_out,
            train_labels_out,
        ),
        (
            "val",
            VAL_NATIVE_IMAGES,
            VAL_NATIVE_ANN,
            val_images_out,
            val_labels_out,
        ),
    ]

    hardlinks = 0
    copies = 0

    for (
        split,
        source_images_dir,
        source_ann_dir,
        target_images_dir,
        target_labels_dir,
    ) in split_specs:

        images = sorted(
            p
            for p in source_images_dir.iterdir()
            if p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTENSIONS
        )

        for source_image in tqdm(
            images,
            desc=f"Preparing {split}",
        ):
            with Image.open(
                source_image
            ) as im:
                width, height = im.size

            annotation_path = (
                source_ann_dir
                / f"{source_image.stem}.txt"
            )

            boxes = native_person_boxes(
                annotation_path,
                width,
                height,
            )

            destination_name = stable_name(
                source_image
            )

            destination_image = (
                target_images_dir
                / destination_name
            )

            destination_label = (
                target_labels_dir
                / Path(
                    destination_name
                ).with_suffix(
                    ".txt"
                ).name
            )

            mode = hardlink_or_copy(
                source_image,
                destination_image,
            )

            if mode == "hardlink":
                hardlinks += 1
            else:
                copies += 1

            write_yolo_boxes(
                destination_label,
                boxes,
            )

            rows.append(
                {
                    "split": split,
                    "source_image": str(
                        source_image
                    ),
                    "image": str(
                        destination_image
                    ),
                    "label": str(
                        destination_label
                    ),
                    "augmentation": "full_frame",
                    "person_boxes": len(
                        boxes
                    ),
                    "is_negative": len(
                        boxes
                    )
                    == 0,
                }
            )

            if (
                split == "train"
                and boxes
            ):
                sides = [
                    min(
                        box[2]
                        * width,
                        box[3]
                        * height,
                    )
                    for box in boxes
                ]

                tiny_indices = [
                    index
                    for index, side
                    in enumerate(
                        sides
                    )
                    if side
                    <= TINY_SIDE_THRESHOLD_PX
                ]

                train_candidates.append(
                    {
                        "source_image": source_image,
                        "boxes": boxes,
                        "tiny_indices": tiny_indices,
                    }
                )

    rng = random.Random(
        SEED
    )

    eligible = [
        item
        for item
        in train_candidates
        if item[
            "tiny_indices"
        ]
    ]

    rng.shuffle(
        eligible
    )

    context_tiles_created = 0
    candidate_cursor = 0

    while (
        eligible
        and context_tiles_created
        < ATPC_CONTEXT_TILES
    ):
        candidate = eligible[
            candidate_cursor
            % len(
                eligible
            )
        ]

        selected_index = (
            candidate[
                "tiny_indices"
            ][
                context_tiles_created
                % len(
                    candidate[
                        "tiny_indices"
                    ]
                )
            ]
        )

        tile_name = (
            f"atpc_"
            f"{context_tiles_created:05d}_"
            f"{stable_name(candidate['source_image'])}"
        )

        tile_name = str(
            Path(
                tile_name
            ).with_suffix(
                ".jpg"
            )
        )

        destination_image = (
            train_images_out
            / tile_name
        )

        destination_label = (
            train_labels_out
            / Path(
                tile_name
            ).with_suffix(
                ".txt"
            ).name
        )

        tile_record = create_context_tile(
            candidate[
                "source_image"
            ],
            candidate[
                "boxes"
            ],
            selected_index,
            destination_image,
            destination_label,
            rng,
        )

        if tile_record is not None:
            rows.append(
                {
                    "split": "train",
                    "source_image": str(
                        candidate[
                            "source_image"
                        ]
                    ),
                    "image": str(
                        destination_image
                    ),
                    "label": str(
                        destination_label
                    ),
                    "augmentation": "tiny_context_tile",
                    "person_boxes": tile_record[
                        "person_boxes"
                    ],
                    "is_negative": False,
                }
            )

            context_tiles_created += 1

        candidate_cursor += 1

        if (
            candidate_cursor
            > ATPC_CONTEXT_TILES
            * 5
        ):
            break

    manifest = pd.DataFrame(
        rows
    )

    manifest.to_csv(
        DATASET_ROOT
        / "manifest.csv",
        index=False,
    )

    data_yaml = {
        "path": str(
            DATASET_ROOT
        ),
        "train": "images/train",
        "val": "images/val",
        "names": {
            0: "person"
        },
        "nc": 1,
    }

    (
        DATASET_ROOT
        / "data.yaml"
    ).write_text(
        yaml.safe_dump(
            data_yaml,
            sort_keys=False,
        ),
        encoding="utf-8",
    )

    print(
        "Hardlinks:",
        hardlinks,
    )

    print(
        "Copies:",
        copies,
    )

    print(
        "ATPC tiles:",
        context_tiles_created,
    )

    return manifest


manifest = build_dataset()

DATA_YAML = (
    DATASET_ROOT
    / "data.yaml"
)

print(
    "\nDataset YAML:\n"
    + DATA_YAML.read_text(
        encoding="utf-8"
    )
)

## 9. Strict dataset and label-path audit

In [ ]:
train_rows = manifest[
    manifest["split"]
    == "train"
]

val_rows = manifest[
    manifest["split"]
    == "val"
]

full_train_count = int(
    (
        train_rows[
            "augmentation"
        ]
        == "full_frame"
    ).sum()
)

tile_count = int(
    (
        train_rows[
            "augmentation"
        ]
        == "tiny_context_tile"
    ).sum()
)

if full_train_count != 6471:
    raise RuntimeError(
        f"Expected 6471 full-frame train images, found {full_train_count}."
    )

if tile_count != 2800:
    raise RuntimeError(
        f"Expected 2800 ATPC tiles, found {tile_count}."
    )

if len(train_rows) != 9271:
    raise RuntimeError(
        f"Expected 9271 total training images, found {len(train_rows)}."
    )

if len(val_rows) != 548:
    raise RuntimeError(
        f"Expected 548 validation images, found {len(val_rows)}."
    )


train_image_paths = [
    str(p)
    for p
    in sorted(
        (
            DATASET_ROOT
            / "images"
            / "train"
        ).iterdir()
    )
    if p.is_file()
    and p.suffix.lower()
    in IMAGE_EXTENSIONS
]

val_image_paths = [
    str(p)
    for p
    in sorted(
        (
            DATASET_ROOT
            / "images"
            / "val"
        ).iterdir()
    )
    if p.is_file()
    and p.suffix.lower()
    in IMAGE_EXTENSIONS
]


resolved_train_labels = [
    Path(p)
    for p
    in img2label_paths(
        train_image_paths
    )
]

resolved_val_labels = [
    Path(p)
    for p
    in img2label_paths(
        val_image_paths
    )
]


missing_train = [
    p
    for p
    in resolved_train_labels
    if not p.exists()
]

missing_val = [
    p
    for p
    in resolved_val_labels
    if not p.exists()
]


if missing_train:
    raise RuntimeError(
        f"Ultralytics cannot resolve {len(missing_train)} train labels. "
        f"Examples: {missing_train[:5]}"
    )

if missing_val:
    raise RuntimeError(
        f"Ultralytics cannot resolve {len(missing_val)} val labels. "
        f"Examples: {missing_val[:5]}"
    )


def validate_label_file(
    path: Path,
):
    errors = []

    for line_number, raw in enumerate(
        path.read_text(
            encoding="utf-8"
        ).splitlines(),
        start=1,
    ):
        raw = raw.strip()

        if not raw:
            continue

        fields = raw.split()

        if len(fields) != 5:
            errors.append(
                f"{path}:{line_number}: expected 5 fields"
            )
            continue

        try:
            cls = int(
                float(
                    fields[0]
                )
            )

            values = [
                float(x)
                for x
                in fields[1:]
            ]

        except Exception:
            errors.append(
                f"{path}:{line_number}: invalid numeric values"
            )
            continue

        if cls != 0:
            errors.append(
                f"{path}:{line_number}: class must be 0"
            )

        if not all(
            0.0 <= v <= 1.0
            for v
            in values
        ):
            errors.append(
                f"{path}:{line_number}: normalized coordinate outside [0,1]"
            )

        if (
            values[2] <= 0
            or values[3] <= 0
        ):
            errors.append(
                f"{path}:{line_number}: invalid width/height"
            )

    return errors


all_errors = []

for label_path in tqdm(
    resolved_train_labels
    + resolved_val_labels,
    desc="Auditing YOLO labels",
):
    all_errors.extend(
        validate_label_file(
            label_path
        )
    )

    if len(all_errors) >= 30:
        break


if all_errors:
    raise RuntimeError(
        "Dataset label audit failed:\n"
        + "\n".join(
            all_errors[:30]
        )
    )


dataset_summary = (
    manifest
    .groupby(
        [
            "split",
            "augmentation",
        ]
    )
    .agg(
        images=(
            "image",
            "count",
        ),
        person_boxes=(
            "person_boxes",
            "sum",
        ),
        negative_images=(
            "is_negative",
            "sum",
        ),
    )
    .reset_index()
)

display(
    dataset_summary
)

dataset_summary.to_csv(
    REPORTS_DIR
    / "dataset_summary.csv",
    index=False,
)


print("\n" + "=" * 88)
print("YOLO26s DATASET AUDIT: PASSED")
print("=" * 88)
print("Official train full frames :", full_train_count)
print("ATPC train tiles           :", tile_count)
print("Total train images         :", len(train_rows))
print("Validation images          :", len(val_rows))
print("Train labels resolved      :", len(resolved_train_labels))
print("Val labels resolved        :", len(resolved_val_labels))
print("Class                      : person")
print("Private Final Test used    : False")
print("=" * 88)

## 10. Build COCO validation ground truth for size-aware metrics

This is generated from the official 548-image VisDrone validation split only.

In [ ]:
COCO_VAL_JSON = (
    METRICS_DIR
    / "visdrone_val_person_coco.json"
)

coco_images = []
coco_annotations = []

annotation_id = 1

val_source_images = sorted(
    p
    for p in VAL_NATIVE_IMAGES.iterdir()
    if p.is_file()
    and p.suffix.lower()
    in IMAGE_EXTENSIONS
)

for image_id, source_image in enumerate(
    val_source_images,
    start=1,
):
    with Image.open(
        source_image
    ) as im:
        width, height = im.size

    boxes = native_person_boxes(
        VAL_NATIVE_ANN
        / f"{source_image.stem}.txt",
        width,
        height,
    )

    destination_name = stable_name(
        source_image
    )

    coco_images.append(
        {
            "id": image_id,
            "file_name": destination_name,
            "width": width,
            "height": height,
        }
    )

    for box in boxes:
        x, y, w, h = box

        bw = (
            w
            * width
        )

        bh = (
            h
            * height
        )

        x1 = (
            x
            * width
            - bw / 2
        )

        y1 = (
            y
            * height
            - bh / 2
        )

        coco_annotations.append(
            {
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 0,
                "bbox": [
                    x1,
                    y1,
                    bw,
                    bh,
                ],
                "area": bw * bh,
                "iscrowd": 0,
                "segmentation": [],
            }
        )

        annotation_id += 1


COCO_VAL_JSON.write_text(
    json.dumps(
        {
            "images": coco_images,
            "annotations": coco_annotations,
            "categories": [
                {
                    "id": 0,
                    "name": "person",
                    "supercategory": "person",
                }
            ],
        },
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


coco_gt_test = COCO(
    str(
        COCO_VAL_JSON
    )
)

if len(
    coco_gt_test.getImgIds()
) != 548:
    raise RuntimeError(
        "COCO validation ground truth must contain 548 images."
    )

print(
    "COCO validation ground truth:",
    COCO_VAL_JSON,
)

print(
    "COCO GT verification: PASSED"
)

## 11. Initialize the new Drive run

The old run is never overwritten.

If a resumable epoch checkpoint exists, the notebook copies the 35-row history into the new run and patches only the copied checkpoint's run metadata and total epoch target to 45.

If optimizer state is unavailable, the new run uses a separate 10-epoch finishing stage and later merges both histories.

In [ ]:
RUN_MODE_FILE = (
    NEW_ROOT
    / "run_mode.json"
)

EXACT_RESUME_LAST = (
    NEW_WEIGHTS_DIR
    / "last.pt"
)

EXACT_RESUME_BEST = (
    NEW_WEIGHTS_DIR
    / "best.pt"
)

CONTINUATION_RUN_DIR = (
    NEW_ROOT
    / "continuation_10e"
)

CONTINUATION_RESULTS = (
    CONTINUATION_RUN_DIR
    / "results.csv"
)

CONTINUATION_LAST = (
    CONTINUATION_RUN_DIR
    / "weights"
    / "last.pt"
)

CONTINUATION_BEST = (
    CONTINUATION_RUN_DIR
    / "weights"
    / "best.pt"
)


def initialize_exact_resume():
    NEW_RUN_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    NEW_WEIGHTS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Preserve the complete first 35 rows in the new run.
    if not (
        NEW_RUN_DIR
        / "results.csv"
    ).exists():
        shutil.copy2(
            OLD_RESULTS_CSV,
            NEW_RUN_DIR
            / "results.csv",
        )

    # Preserve the best checkpoint from epochs 1–35.
    if not EXACT_RESUME_BEST.exists():
        shutil.copy2(
            OLD_BEST,
            EXACT_RESUME_BEST,
        )

    # Do not overwrite a newer continuation last.pt on rerun.
    existing_history = pd.read_csv(
        NEW_RUN_DIR
        / "results.csv"
    )

    if (
        len(existing_history)
        > OLD_EPOCHS
        and EXACT_RESUME_LAST.exists()
    ):
        print(
            "Existing partially continued exact-resume run detected."
        )

        return EXACT_RESUME_LAST

    source_state = torch.load(
        SOURCE_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    train_args = dict(
        source_state.get(
            "train_args",
            {},
        )
    )

    train_args.update(
        {
            "data": str(
                DATA_YAML
            ),
            "epochs": TOTAL_EPOCHS,
            "imgsz": IMAGE_SIZE,
            "batch": BATCH_SIZE,
            "device": 0,
            "workers": WORKERS,
            "save_period": 1,
            "close_mosaic": 10,
            "cache": False,
            "patience": 100,
            "plots": True,
            "val": True,
            "project": str(
                NEW_ROOT
            ),
            "name": NEW_RUN_DIR.name,
            "exist_ok": True,
            "save_dir": str(
                NEW_RUN_DIR
            ),
        }
    )

    source_state[
        "train_args"
    ] = train_args

    # Patch model/EMA args as well when present.
    model_obj = source_state.get(
        "model"
    )

    if (
        model_obj is not None
        and hasattr(
            model_obj,
            "args",
        )
    ):
        model_obj.args = {
            **getattr(
                model_obj,
                "args",
                {},
            ),
            **train_args,
        }

    ema_obj = source_state.get(
        "ema"
    )

    if (
        ema_obj is not None
        and hasattr(
            ema_obj,
            "args",
        )
    ):
        ema_obj.args = {
            **getattr(
                ema_obj,
                "args",
                {},
            ),
            **train_args,
        }

    torch.save(
        source_state,
        EXACT_RESUME_LAST,
    )

    del source_state
    gc.collect()

    return EXACT_RESUME_LAST


if CONTINUATION_MODE == "exact_optimizer_resume":
    ACTIVE_CHECKPOINT = initialize_exact_resume()
    ACTIVE_RESULTS_CSV = NEW_RUN_DIR / "results.csv"
    ACTIVE_RUN_DIR = NEW_RUN_DIR
    INITIAL_GLOBAL_ROWS = OLD_EPOCHS

else:
    ACTIVE_CHECKPOINT = OLD_LAST
    ACTIVE_RESULTS_CSV = CONTINUATION_RESULTS
    ACTIVE_RUN_DIR = CONTINUATION_RUN_DIR
    INITIAL_GLOBAL_ROWS = OLD_EPOCHS


RUN_MODE_FILE.write_text(
    json.dumps(
        {
            "mode": CONTINUATION_MODE,
            "source_checkpoint": str(
                SOURCE_CHECKPOINT
            ),
            "active_checkpoint": str(
                ACTIVE_CHECKPOINT
            ),
            "active_run_dir": str(
                ACTIVE_RUN_DIR
            ),
            "old_epochs": OLD_EPOCHS,
            "additional_epochs": ADDITIONAL_EPOCHS,
            "target_total_epochs": TOTAL_EPOCHS,
        },
        indent=2,
    ),
    encoding="utf-8",
)


print("=" * 88)
print("NEW RUN INITIALIZATION")
print("=" * 88)
print("Mode       :", CONTINUATION_MODE)
print("Active run :", ACTIVE_RUN_DIR)
print("Old run    :", OLD_RUN_DIR)
print("New root   :", NEW_ROOT)
print("=" * 88)

## 12. Write the continuation launcher

The launcher supports both:
- exact optimizer-state resume to epoch 45, or
- a safe 10-epoch finishing fine-tune from the old `last.pt` if the completed run no longer contains resumable optimizer state.

In [ ]:
LAUNCHER_PATH = Path(
    "/content/continue_yolo26s_35_to_45.py"
)

launcher_code = f'''
from pathlib import Path
import pandas as pd
import torch
from ultralytics import YOLO

MODE = {CONTINUATION_MODE!r}
DATA_YAML = Path({str(DATA_YAML)!r})
NEW_RUN_DIR = Path({str(NEW_RUN_DIR)!r})
CONTINUATION_RUN_DIR = Path({str(CONTINUATION_RUN_DIR)!r})
EXACT_RESUME_LAST = Path({str(EXACT_RESUME_LAST)!r})
OLD_LAST = Path({str(OLD_LAST)!r})

if MODE == "exact_optimizer_resume":
    model = YOLO(str(EXACT_RESUME_LAST))
    model.train(
        resume=True,
        data=str(DATA_YAML),
        imgsz={IMAGE_SIZE},
        batch={BATCH_SIZE},
        device=0,
        workers={WORKERS},
        save_period=1,
        close_mosaic=10,
        cache=False,
        patience=100,
        plots=True,
        val=True,
        save_dir=str(NEW_RUN_DIR),
    )

else:
    if (CONTINUATION_RUN_DIR / "weights" / "last.pt").exists():
        existing = (
            pd.read_csv(CONTINUATION_RUN_DIR / "results.csv")
            if (CONTINUATION_RUN_DIR / "results.csv").exists()
            else pd.DataFrame()
        )

        if len(existing) < {ADDITIONAL_EPOCHS}:
            model = YOLO(
                str(
                    CONTINUATION_RUN_DIR
                    / "weights"
                    / "last.pt"
                )
            )

            model.train(
                resume=True,
            )

        else:
            print(
                "Fallback continuation is already complete."
            )

    else:
        # The old completed last.pt is used as the requested source model.
        # This is a finishing fine-tune, not an optimizer-state resume.
        model = YOLO(str(OLD_LAST))

        model.train(
            data=str(DATA_YAML),
            epochs={ADDITIONAL_EPOCHS},
            imgsz={IMAGE_SIZE},
            batch={BATCH_SIZE},
            device=0,
            workers={WORKERS},
            project=str(CONTINUATION_RUN_DIR.parent),
            name=CONTINUATION_RUN_DIR.name,
            exist_ok=True,
            pretrained=True,
            optimizer="AdamW",
            lr0=0.00005,
            lrf=0.10,
            weight_decay=0.0005,
            cos_lr=True,
            warmup_epochs=0.5,
            patience=100,
            save=True,
            save_period=1,
            seed={SEED},
            deterministic=True,
            amp={USE_AMP},
            plots=True,
            val=True,
            single_cls=True,
            fraction=1.0,
            mosaic=0.0,
            close_mosaic=0,
            mixup=0.0,
            degrees=5.0,
            translate=0.10,
            scale=0.25,
            shear=1.0,
            perspective=0.0001,
            fliplr=0.50,
            flipud=0.0,
            hsv_h=0.015,
            hsv_s=0.30,
            hsv_v=0.25,
            cache=False,
            verbose=True,
        )
'''

LAUNCHER_PATH.write_text(
    launcher_code,
    encoding="utf-8",
)

compile(
    launcher_code,
    str(
        LAUNCHER_PATH
    ),
    "exec",
)

print(
    "Launcher:",
    LAUNCHER_PATH,
)

print(
    "Launcher syntax: PASSED"
)

## 13. Strong preflight before training

This verifies:
- the new Drive path is different from the old one,
- old 35-row history is intact,
- the dataset has 9271 train / 548 val images,
- the active checkpoint loads as YOLO26,
- the class is `person`,
- the training launcher is syntactically valid.

In [ ]:
if OLD_RUN_DIR.resolve() == NEW_RUN_DIR.resolve():
    raise RuntimeError(
        "The new run directory must not equal the old run directory."
    )


if len(
    pd.read_csv(
        OLD_RESULTS_CSV
    )
) != 35:
    raise RuntimeError(
        "Old 35-epoch history verification failed."
    )


train_image_count = len(
    [
        p
        for p
        in (
            DATASET_ROOT
            / "images"
            / "train"
        ).iterdir()
        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]
)

val_image_count = len(
    [
        p
        for p
        in (
            DATASET_ROOT
            / "images"
            / "val"
        ).iterdir()
        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]
)


if train_image_count != 9271:
    raise RuntimeError(
        f"Unexpected training image count: {train_image_count}"
    )

if val_image_count != 548:
    raise RuntimeError(
        f"Unexpected validation image count: {val_image_count}"
    )


preflight_model = YOLO(
    str(
        ACTIVE_CHECKPOINT
    )
)

model_names = getattr(
    preflight_model.model,
    "names",
    {},
)

print(
    "Loaded model names:",
    model_names,
)

if isinstance(
    model_names,
    dict,
):
    detected_name = str(
        model_names.get(
            0,
            "",
        )
    ).lower()
else:
    detected_name = str(
        model_names[0]
    ).lower()


if detected_name != "person":
    raise RuntimeError(
        f"Expected class 0='person', got: {model_names}"
    )


print("=" * 88)
print("YOLO26s 35→45 PREFLIGHT: PASSED")
print("=" * 88)
print("Mode        :", CONTINUATION_MODE)
print("Source      :", SOURCE_CHECKPOINT)
print("Old history :", "35 rows")
print("Train images:", train_image_count)
print("Val images  :", val_image_count)
print("Input       :", IMAGE_SIZE)
print("Batch       :", BATCH_SIZE)
print("GPU         :", torch.cuda.get_device_name(0))
print("New root    :", NEW_ROOT)
print("=" * 88)

del preflight_model
gc.collect()
torch.cuda.empty_cache()

## 14. Continue epochs 36–45 with clean epoch-end reporting

Normal batch logs are hidden.  
After each newly completed epoch the notebook prints:
- train losses available in YOLO26
- validation Precision
- validation Recall
- validation mAP50
- validation mAP50-95
- epoch wall time
- best mAP50-95 across the full 45-epoch history
- checkpoint status

If Colab is interrupted, rerun the notebook. The last fully completed new epoch remains resumable.

In [ ]:
def read_csv_safe(
    path: Path,
):
    if not path.exists():
        return pd.DataFrame()

    try:
        df = pd.read_csv(
            path
        )

        df.columns = [
            str(c).strip()
            for c
            in df.columns
        ]

        return df

    except Exception:
        return pd.DataFrame()


def find_metric(
    row,
    candidates,
):
    for key in row.index:
        normalized = (
            str(key)
            .strip()
            .lower()
            .replace(
                " ",
                "",
            )
        )

        if any(
            candidate
            in normalized
            for candidate
            in candidates
        ):
            try:
                return float(
                    row[key]
                )
            except Exception:
                return None

    return None


def fmt(
    value,
):
    return (
        "n/a"
        if value is None
        or not np.isfinite(
            value
        )
        else f"{value:.6f}"
    )


old_best_so_far = old_best_map

if CONTINUATION_MODE == "exact_optimizer_resume":
    existing_new_rows = max(
        0,
        len(
            read_csv_safe(
                ACTIVE_RESULTS_CSV
            )
        )
        - OLD_EPOCHS,
    )
else:
    existing_new_rows = len(
        read_csv_safe(
            ACTIVE_RESULTS_CSV
        )
    )


print(
    "Already completed continuation epochs:",
    existing_new_rows,
    "/",
    ADDITIONAL_EPOCHS,
)


if existing_new_rows >= ADDITIONAL_EPOCHS:
    print(
        "The 10 additional epochs are already complete."
    )

else:
    stop_reporter = threading.Event()
    report_lock = threading.Lock()

    seen_new_rows = existing_new_rows
    training_start = time.time()
    previous_epoch_end = training_start
    best_map_so_far = old_best_so_far


    def report_new_epochs():
        global seen_new_rows
        global previous_epoch_end
        global best_map_so_far

        df = read_csv_safe(
            ACTIVE_RESULTS_CSV
        )

        if CONTINUATION_MODE == "exact_optimizer_resume":
            continuation_df = df.iloc[
                OLD_EPOCHS:
            ].reset_index(
                drop=True
            )
        else:
            continuation_df = df.reset_index(
                drop=True
            )

        if len(
            continuation_df
        ) <= seen_new_rows:
            return

        for local_index in range(
            seen_new_rows,
            len(
                continuation_df
            ),
        ):
            row = continuation_df.iloc[
                local_index
            ]

            global_epoch = (
                OLD_EPOCHS
                + local_index
                + 1
            )

            train_box = find_metric(
                row,
                [
                    "train/box_loss",
                    "box_loss",
                ],
            )

            train_cls = find_metric(
                row,
                [
                    "train/cls_loss",
                    "cls_loss",
                ],
            )

            train_l1 = find_metric(
                row,
                [
                    "train/l1_loss",
                    "l1_loss",
                ],
            )

            precision = find_metric(
                row,
                [
                    "metrics/precision(b)",
                    "metrics/precision",
                ],
            )

            recall = find_metric(
                row,
                [
                    "metrics/recall(b)",
                    "metrics/recall",
                ],
            )

            map50 = find_metric(
                row,
                [
                    "metrics/map50(b)",
                    "metrics/map50",
                ],
            )

            map5095 = find_metric(
                row,
                [
                    "metrics/map50-95(b)",
                    "metrics/map50-95",
                ],
            )

            if (
                map5095 is not None
                and np.isfinite(
                    map5095
                )
            ):
                best_map_so_far = max(
                    best_map_so_far,
                    map5095,
                )

            now = time.time()

            epoch_seconds = (
                now
                - previous_epoch_end
            )

            previous_epoch_end = now

            if CONTINUATION_MODE == "exact_optimizer_resume":
                checkpoint_path = (
                    NEW_WEIGHTS_DIR
                    / "last.pt"
                )
            else:
                checkpoint_path = (
                    CONTINUATION_RUN_DIR
                    / "weights"
                    / "last.pt"
                )

            print(
                "\n"
                + "=" * 72,
                flush=True,
            )

            print(
                f"EPOCH {global_epoch:02d} / "
                f"{TOTAL_EPOCHS:02d} COMPLETED",
                flush=True,
            )

            print(
                "=" * 72,
                flush=True,
            )

            print(
                f"Train Box Loss      : {fmt(train_box)}",
                flush=True,
            )

            print(
                f"Train Class Loss    : {fmt(train_cls)}",
                flush=True,
            )

            if train_l1 is not None:
                print(
                    f"Train L1 Loss       : {fmt(train_l1)}",
                    flush=True,
                )

            print(
                f"Validation Precision: {fmt(precision)}",
                flush=True,
            )

            print(
                f"Validation Recall   : {fmt(recall)}",
                flush=True,
            )

            print(
                f"Validation mAP50    : {fmt(map50)}",
                flush=True,
            )

            print(
                f"Validation mAP50-95 : {fmt(map5095)}",
                flush=True,
            )

            print(
                f"Best mAP50-95       : {fmt(best_map_so_far)}",
                flush=True,
            )

            print(
                "Epoch Time          :",
                time.strftime(
                    "%H:%M:%S",
                    time.gmtime(
                        epoch_seconds
                    ),
                ),
                flush=True,
            )

            print(
                "Total Session Time  :",
                time.strftime(
                    "%H:%M:%S",
                    time.gmtime(
                        now
                        - training_start
                    ),
                ),
                flush=True,
            )

            print(
                "Checkpoint          :",
                (
                    "SAVED"
                    if checkpoint_path.exists()
                    else "WAITING"
                ),
                flush=True,
            )

            print(
                "=" * 72
                + "\n",
                flush=True,
            )

        seen_new_rows = len(
            continuation_df
        )


    def reporter_loop():
        while not stop_reporter.is_set():
            with report_lock:
                report_new_epochs()

            stop_reporter.wait(
                3
            )


    env = os.environ.copy()
    env[
        "PYTHONUNBUFFERED"
    ] = "1"
    env[
        "CUDA_VISIBLE_DEVICES"
    ] = "0"

    recent_output = deque(
        maxlen=250
    )

    reporter = threading.Thread(
        target=reporter_loop,
        daemon=True,
    )

    reporter.start()

    process = subprocess.Popen(
        [
            sys.executable,
            "-u",
            str(
                LAUNCHER_PATH
            ),
        ],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    manual_stop = False

    try:
        for line in process.stdout:
            clean = line.rstrip()

            recent_output.append(
                clean
            )

            low = clean.lower()

            critical_patterns = (
                "traceback",
                "runtimeerror",
                "cuda out of memory",
                "filenotfounderror",
                "modulenotfounderror",
                "fatal",
                "nan detected",
            )

            if any(
                pattern in low
                for pattern
                in critical_patterns
            ):
                print(
                    clean,
                    flush=True,
                )

        return_code = process.wait()

    except KeyboardInterrupt:
        manual_stop = True

        print(
            "\nManual stop requested. "
            "Stopping the child training process..."
        )

        process.terminate()

        try:
            process.wait(
                timeout=20
            )

        except subprocess.TimeoutExpired:
            process.kill()

    finally:
        time.sleep(
            2
        )

        with report_lock:
            report_new_epochs()

        stop_reporter.set()
        reporter.join(
            timeout=10
        )


    if manual_stop:
        print(
            "\nSAFE STOP COMPLETE."
        )

        print(
            "Rerun this notebook to continue from "
            "the last fully completed new epoch."
        )

    elif return_code != 0:
        print(
            "\nLast diagnostic lines:"
        )

        for line in recent_output:
            print(
                line
            )

        raise RuntimeError(
            f"YOLO26s continuation exited with code {return_code}."
        )

    else:
        print(
            "\nContinuation process returned successfully."
        )

## 15. Build the complete 45-epoch history and choose the final best checkpoint

This cell always creates a 45-row combined history when all additional epochs are complete.

In [ ]:
old_df = read_csv_safe(
    OLD_RESULTS_CSV
)

if len(old_df) != OLD_EPOCHS:
    raise RuntimeError(
        f"Old history is not {OLD_EPOCHS} rows."
    )


if CONTINUATION_MODE == "exact_optimizer_resume":
    resumed_df = read_csv_safe(
        NEW_RUN_DIR
        / "results.csv"
    )

    if len(resumed_df) < TOTAL_EPOCHS:
        raise RuntimeError(
            f"Training is incomplete: "
            f"{len(resumed_df)}/{TOTAL_EPOCHS} rows available."
        )

    combined_df = resumed_df.iloc[
        :TOTAL_EPOCHS
    ].copy()

    combined_df.insert(
        0,
        "global_epoch",
        range(
            1,
            TOTAL_EPOCHS
            + 1,
        ),
    )

    combined_df.insert(
        1,
        "phase",
        [
            "original_35e"
            if epoch
            <= OLD_EPOCHS
            else "continued_10e"
            for epoch
            in range(
                1,
                TOTAL_EPOCHS
                + 1,
            )
        ],
    )

    final_run_best = (
        NEW_WEIGHTS_DIR
        / "best.pt"
    )

    final_run_last = (
        NEW_WEIGHTS_DIR
        / "last.pt"
    )

else:
    continuation_df = read_csv_safe(
        CONTINUATION_RESULTS
    )

    if len(
        continuation_df
    ) < ADDITIONAL_EPOCHS:
        raise RuntimeError(
            f"Training is incomplete: "
            f"{len(continuation_df)}/{ADDITIONAL_EPOCHS} "
            "continuation rows available."
        )

    old_part = old_df.copy()

    old_part.insert(
        0,
        "global_epoch",
        range(
            1,
            OLD_EPOCHS
            + 1,
        ),
    )

    old_part.insert(
        1,
        "phase",
        "original_35e",
    )

    new_part = (
        continuation_df
        .iloc[
            :ADDITIONAL_EPOCHS
        ]
        .copy()
    )

    new_part.insert(
        0,
        "global_epoch",
        range(
            OLD_EPOCHS
            + 1,
            TOTAL_EPOCHS
            + 1,
        ),
    )

    new_part.insert(
        1,
        "phase",
        "continued_10e",
    )

    combined_df = pd.concat(
        [
            old_part,
            new_part,
        ],
        ignore_index=True,
        sort=False,
    )

    continuation_map_col = next(
        (
            c
            for c
            in continuation_df.columns
            if "map50-95"
            in c.lower()
        ),
        None,
    )

    if continuation_map_col is None:
        raise RuntimeError(
            "Continuation mAP50-95 column was not found."
        )

    continuation_best_map = float(
        pd.to_numeric(
            continuation_df[
                continuation_map_col
            ],
            errors="coerce",
        ).max()
    )

    if continuation_best_map > old_best_map:
        final_run_best = (
            CONTINUATION_BEST
        )
    else:
        final_run_best = (
            OLD_BEST
        )

    final_run_last = (
        CONTINUATION_LAST
    )


if len(combined_df) != TOTAL_EPOCHS:
    raise RuntimeError(
        f"Combined history has {len(combined_df)} rows, "
        f"expected {TOTAL_EPOCHS}."
    )


COMBINED_HISTORY_CSV = (
    HISTORY_DIR
    / "training_history_45_epochs.csv"
)

combined_df.to_csv(
    COMBINED_HISTORY_CSV,
    index=False,
)


if not final_run_best.exists():
    raise FileNotFoundError(
        final_run_best
    )

if not final_run_last.exists():
    raise FileNotFoundError(
        final_run_last
    )


FINAL_BEST = (
    FINAL_WEIGHTS_DIR
    / "best.pt"
)

FINAL_LAST = (
    FINAL_WEIGHTS_DIR
    / "last.pt"
)

shutil.copy2(
    final_run_best,
    FINAL_BEST,
)

shutil.copy2(
    final_run_last,
    FINAL_LAST,
)


combined_map_col = next(
    (
        c
        for c
        in combined_df.columns
        if "map50-95"
        in c.lower()
    ),
    None,
)

combined_map_values = pd.to_numeric(
    combined_df[
        combined_map_col
    ],
    errors="coerce",
)

best_index = int(
    combined_map_values.idxmax()
)

best_global_epoch = int(
    combined_df.iloc[
        best_index
    ][
        "global_epoch"
    ]
)

best_45_map = float(
    combined_map_values.max()
)


print("=" * 88)
print("45-EPOCH HISTORY: COMPLETE")
print("=" * 88)
print("Rows              :", len(combined_df))
print("Best global epoch :", best_global_epoch)
print("Best mAP50-95     :", f"{best_45_map:.6f}")
print("Combined history  :", COMBINED_HISTORY_CSV)
print("Final best        :", FINAL_BEST)
print("Final last        :", FINAL_LAST)
print("=" * 88)

## 16. Save complete 45-epoch training curves

In [ ]:
def metric_column(
    dataframe,
    fragments,
):
    for column in dataframe.columns:
        normalized = (
            str(
                column
            )
            .lower()
            .replace(
                " ",
                "",
            )
        )

        if any(
            fragment
            in normalized
            for fragment
            in fragments
        ):
            return column

    return None


plot_specs = [
    (
        "train_box_loss",
        [
            "train/box_loss",
        ],
    ),
    (
        "train_cls_loss",
        [
            "train/cls_loss",
        ],
    ),
    (
        "train_l1_loss",
        [
            "train/l1_loss",
        ],
    ),
    (
        "precision",
        [
            "metrics/precision(b)",
            "metrics/precision",
        ],
    ),
    (
        "recall",
        [
            "metrics/recall(b)",
            "metrics/recall",
        ],
    ),
    (
        "mAP50",
        [
            "metrics/map50(b)",
            "metrics/map50",
        ],
    ),
    (
        "mAP50-95",
        [
            "metrics/map50-95(b)",
            "metrics/map50-95",
        ],
    ),
]


for display_name, fragments in plot_specs:
    column = metric_column(
        combined_df,
        fragments,
    )

    if column is None:
        continue

    values = pd.to_numeric(
        combined_df[
            column
        ],
        errors="coerce",
    )

    figure = plt.figure(
        figsize=(
            9,
            5,
        )
    )

    plt.plot(
        combined_df[
            "global_epoch"
        ],
        values,
        marker="o",
        markersize=3,
    )

    plt.axvline(
        OLD_EPOCHS,
        linestyle="--",
    )

    plt.xlabel(
        "Global Epoch"
    )

    plt.ylabel(
        display_name
    )

    plt.title(
        f"YOLO26s 45-Epoch History — {display_name}"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    output_path = (
        HISTORY_DIR
        / f"{display_name}_45e.png"
    )

    figure.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.show()

    plt.close(
        figure
    )


print(
    "Training curve images saved to:",
    HISTORY_DIR,
)

## 17. Final Ultralytics validation

This evaluates the selected final best checkpoint on the official 548-image VisDrone validation split.

In [ ]:
final_model = YOLO(
    str(
        FINAL_BEST
    )
)

FINAL_VAL_DIR = (
    REPORTS_DIR
    / "final_visdrone_val"
)

metrics = final_model.val(
    data=str(
        DATA_YAML
    ),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=1,
    device=0,
    conf=0.001,
    iou=0.70,
    max_det=1000,
    plots=True,
    save_json=False,
    project=str(
        REPORTS_DIR
    ),
    name="final_visdrone_val",
    exist_ok=True,
    verbose=True,
)


ultralytics_metrics = {
    "precision": float(
        metrics.box.mp
    ),
    "recall": float(
        metrics.box.mr
    ),
    "map50": float(
        metrics.box.map50
    ),
    "map50_95": float(
        metrics.box.map
    ),
    "preprocess_ms": float(
        metrics.speed.get(
            "preprocess",
            np.nan,
        )
    ),
    "inference_ms": float(
        metrics.speed.get(
            "inference",
            np.nan,
        )
    ),
    "postprocess_ms": float(
        metrics.speed.get(
            "postprocess",
            np.nan,
        )
    ),
    "checkpoint": str(
        FINAL_BEST
    ),
    "imgsz": IMAGE_SIZE,
    "split": "VisDrone val",
}


(
    METRICS_DIR
    / "ultralytics_final_validation.json"
).write_text(
    json.dumps(
        ultralytics_metrics,
        indent=2,
    ),
    encoding="utf-8",
)


display(
    pd.DataFrame(
        [
            ultralytics_metrics
        ]
    )
)

## 18. Standardized COCO size-aware metrics

This produces the same core metrics needed for comparison with RT-DETR and BPD:
- mAP50-95
- AP50
- AP75
- AP Small
- AP Medium
- AP Large
- AR100

In [ ]:
coco_gt = COCO(
    str(
        COCO_VAL_JSON
    )
)

file_name_to_image_id = {
    image[
        "file_name"
    ]: image[
        "id"
    ]
    for image
    in coco_gt.dataset[
        "images"
    ]
}


val_images_for_prediction = sorted(
    p
    for p
    in (
        DATASET_ROOT
        / "images"
        / "val"
    ).iterdir()
    if p.is_file()
    and p.suffix.lower()
    in IMAGE_EXTENSIONS
)


prediction_rows = []

prediction_results = final_model.predict(
    source=[
        str(p)
        for p
        in val_images_for_prediction
    ],
    imgsz=IMAGE_SIZE,
    conf=0.001,
    iou=0.70,
    max_det=1000,
    device=0,
    verbose=False,
    stream=True,
)


for result in tqdm(
    prediction_results,
    total=len(
        val_images_for_prediction
    ),
    desc="COCO prediction export",
):
    file_name = Path(
        result.path
    ).name

    image_id = file_name_to_image_id.get(
        file_name
    )

    if image_id is None:
        raise RuntimeError(
            f"Prediction image was not found in COCO GT: {file_name}"
        )

    boxes = result.boxes

    if (
        boxes is None
        or len(
            boxes
        )
        == 0
    ):
        continue

    xyxy = (
        boxes.xyxy
        .detach()
        .cpu()
        .numpy()
    )

    scores = (
        boxes.conf
        .detach()
        .cpu()
        .numpy()
    )

    for box, score in zip(
        xyxy,
        scores,
    ):
        x1, y1, x2, y2 = [
            float(v)
            for v
            in box
        ]

        prediction_rows.append(
            {
                "image_id": int(
                    image_id
                ),
                "category_id": 0,
                "bbox": [
                    x1,
                    y1,
                    x2 - x1,
                    y2 - y1,
                ],
                "score": float(
                    score
                ),
            }
        )


PREDICTIONS_JSON = (
    METRICS_DIR
    / "visdrone_val_predictions.json"
)

PREDICTIONS_JSON.write_text(
    json.dumps(
        prediction_rows
    ),
    encoding="utf-8",
)


coco_dt = coco_gt.loadRes(
    str(
        PREDICTIONS_JSON
    )
)

evaluator = COCOeval(
    coco_gt,
    coco_dt,
    "bbox",
)

evaluator.params.catIds = [
    0
]

evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

stats = evaluator.stats.tolist()


coco_metrics = {
    "mAP50_95": float(
        stats[0]
    ),
    "AP50": float(
        stats[1]
    ),
    "AP75": float(
        stats[2]
    ),
    "AP_small": float(
        stats[3]
    ),
    "AP_medium": float(
        stats[4]
    ),
    "AP_large": float(
        stats[5]
    ),
    "AR100": float(
        stats[8]
    ),
    "images": 548,
    "class": "person",
    "checkpoint": str(
        FINAL_BEST
    ),
}


(
    METRICS_DIR
    / "standardized_coco_metrics.json"
).write_text(
    json.dumps(
        coco_metrics,
        indent=2,
    ),
    encoding="utf-8",
)


print("\n" + "=" * 72)
print("FINAL STANDARDIZED COCO METRICS")
print("=" * 72)

for key, value in coco_metrics.items():
    print(
        f"{key:12s}:",
        value,
    )

print("=" * 72)

## 19. Final audit report, hashes, and artifact manifest

In [ ]:
final_report = {
    "model": "YOLO26s",
    "architecture": "official YOLO26s",
    "training_history": {
        "source_epochs": OLD_EPOCHS,
        "additional_epochs": ADDITIONAL_EPOCHS,
        "total_epochs": TOTAL_EPOCHS,
        "continuation_mode": CONTINUATION_MODE,
        "best_global_epoch": best_global_epoch,
        "best_map50_95_from_history": best_45_map,
    },
    "source_run": {
        "old_run_dir": str(
            OLD_RUN_DIR
        ),
        "source_checkpoint": str(
            SOURCE_CHECKPOINT
        ),
        "old_best_map50_95": old_best_map,
    },
    "data": {
        "dataset": "VisDrone2019-DET",
        "target_class": "person",
        "official_train_full_frames": 6471,
        "atpc_train_tiles": 2800,
        "total_train_images": 9271,
        "validation_images": 548,
        "input_size": IMAGE_SIZE,
        "private_final_test_used": False,
    },
    "training": {
        "batch": BATCH_SIZE,
        "seed": SEED,
        "gpu": torch.cuda.get_device_name(
            0
        ),
        "amp": USE_AMP,
        "ultralytics_version": __import__(
            "ultralytics"
        ).__version__,
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
    },
    "final_checkpoints": {
        "best": str(
            FINAL_BEST
        ),
        "best_sha256": sha256_file(
            FINAL_BEST
        ),
        "last": str(
            FINAL_LAST
        ),
        "last_sha256": sha256_file(
            FINAL_LAST
        ),
    },
    "ultralytics_validation": ultralytics_metrics,
    "coco_validation": coco_metrics,
    "combined_history_csv": str(
        COMBINED_HISTORY_CSV
    ),
    "new_output_root": str(
        NEW_ROOT
    ),
}


FINAL_REPORT_JSON = (
    REPORTS_DIR
    / "YOLO26s_STEP3_45E_FINAL_REPORT.json"
)

FINAL_REPORT_JSON.write_text(
    json.dumps(
        final_report,
        indent=2,
    ),
    encoding="utf-8",
)


manifest_rows = []

for file_path in sorted(
    NEW_ROOT.rglob("*")
):
    if (
        not file_path.is_file()
        or file_path.suffix.lower()
        in {
            ".cache",
        }
    ):
        continue

    relative = file_path.relative_to(
        NEW_ROOT
    )

    manifest_rows.append(
        {
            "path": str(
                relative
            ),
            "bytes": file_path.stat().st_size,
        }
    )


pd.DataFrame(
    manifest_rows
).to_csv(
    REPORTS_DIR
    / "artifact_manifest.csv",
    index=False,
)


print("\n" + "=" * 88)
print("YOLO26s 45-EPOCH CONTINUATION: COMPLETE")
print("=" * 88)
print("New output root :", NEW_ROOT)
print("History         :", COMBINED_HISTORY_CSV)
print("Best checkpoint :", FINAL_BEST)
print("Last checkpoint :", FINAL_LAST)
print("Final report    :", FINAL_REPORT_JSON)
print("COCO metrics    :", METRICS_DIR / "standardized_coco_metrics.json")
print("Private Final Test used: False")
print("=" * 88)

## 20. Resume rule

If Colab is interrupted during epochs 36–45:

1. Wait for a clean `EPOCH XX / 45 COMPLETED` report whenever possible.
2. Stop/disconnect.
3. Do not delete the new `last.pt`.
4. Reopen this notebook and run it from the top.
5. It will detect the already-completed new epochs and continue from the last full checkpoint.

Do not delete the old source run until the 45-epoch continuation and final audit are complete.

### Comparison note
For final project evaluation, keep this continuation run identified as **YOLO26s + ATPC**.  
If a strict apples-to-apples architecture baseline is required against RT-DETR-R18 and BPD-YOLOn/L-FPN, train a separate YOLO26s run using only the same 6471-image VisDrone train split and the same 548-image validation split.